# RepairScope-X — Notebook 06 FINAL — FIXED v2
## Repair Energy, Error Recurrence, and Hysteresis — AR Control

**Fully standalone.** No previous notebook or ZIP is required.

This notebook embeds the exact outputs from Notebook 04 and Notebook 05:

- natural failures: **20**
- semantic/non-format failures used by Notebook 05: **19**
- repair trials: **285**
- Notebook 05 Oracle Repair Headroom: **0.1579**
- best fixed action: **tail_repair**
- best fixed success: **0.3860**
- oracle success: **0.5439**
- oracle action concentration: **0.4737**
- positive headroom on **3 datasets**
- embedded payload SHA-256: `9ec1c92e0a075967d5ea0e4e098e03608f595a7492baedea41f38657bc75b75b`

Notebook 05 therefore found meaningful **repair heterogeneity** in the family-matched AR control. This notebook asks the next question:

> When a repair fails, does it repeatedly return to the **same wrong answer**, and how much compute is required to escape that error state?

### Main quantities

**Repair energy.** For a failure/action pair, we estimate:
- empirical repair probability;
- mean actual forward calls;
- observed minimum successful NFE;
- approximate retry energy `mean NFE / success probability`.

The retry-energy quantity is explicitly an exploratory geometric-retry approximation, not a theorem.

**Error recurrence / hysteresis.** A failed repair is tagged `same_wrong` if its numeric prediction returns to the same wrong value as the original natural failure. We compare high-preservation local actions with broader regeneration.

### Confirmatory extension

The notebook automatically selects **6 mechanistically informative failures** from the pilot, with dataset diversity, and—when `RUN_CONFIRMATORY=True`—runs **7 new seeds** for four actions:

- `tail_repair`
- `suffix_75`
- `suffix_50`
- `fresh_retry`

Together with the existing 3 seeds, this gives **10 trials per selected failure/action**.

`editor_rewrite` is not repeated because Notebook 05 observed 0/57 successes and substantially higher compute; it remains in the exploratory analysis but is omitted from the targeted confirmatory extension.

This remains an **AR control**. Diffusion-specific claims require separate DLM experiments.

**v2 fix:** robustly recovers the legacy malformed `confirmatory_trials.jsonl` and writes proper JSONL newlines.

In [ ]:
!pip -q install -U "transformers>=4.48,<5" "accelerate>=1.0" pandas matplotlib

## 1. Recover and validate the embedded upstream results

In [ ]:
import base64, zlib, json, hashlib, math, re, random, time, gc, os, platform, difflib
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd

EMBEDDED_PAYLOAD_SHA256 = "9ec1c92e0a075967d5ea0e4e098e03608f595a7492baedea41f38657bc75b75b"
_payload_raw = zlib.decompress(base64.b64decode('''...'''))

assert hashlib.sha256(_payload_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_payload_raw.decode("utf-8"))
NATURAL_FAILURES = UPSTREAM["natural_failures"]
PILOT_TRIALS = UPSTREAM["repair_trials"]
UPSTREAM_GATE = UPSTREAM["ar_gate"]
UPSTREAM_MANIFEST = UPSTREAM["run_manifest"]

assert len(NATURAL_FAILURES) == 20
assert len(PILOT_TRIALS) == 285
assert UPSTREAM_GATE["AR_REPAIR_HETEROGENEITY_PRESENT"] is True

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)
print("Natural failures:", len(NATURAL_FAILURES))
print("Pilot repair trials:", len(PILOT_TRIALS))
print("Notebook 05 ORH:", UPSTREAM_GATE["oracle_headroom"])
print("Embedded upstream results: PASS")

## 2. Restrict primary analysis to semantic/non-format failures

The single `format_only_candidate` remains available for secondary inspection but is excluded from the primary repair-energy/hysteresis analysis.

In [ ]:
FAILURES = [
    x for x in NATURAL_FAILURES
    if x["verification"]["category"] != "format_only_candidate"
]

FAILURE_MAP = {
    x["problem_id"]: x
    for x in FAILURES
}

PILOT = [
    x for x in PILOT_TRIALS
    if x["problem_id"] in FAILURE_MAP
]

assert len(FAILURES) == 19
assert len(PILOT) == 285

print("Primary failures:", len(FAILURES))
print("Datasets:", dict(Counter(x["dataset"] for x in FAILURES)))

## 3. Numeric state extractor and transition labels

In [ ]:
NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def numeric_state_from_verification(v):
    x = v.get("strict_prediction")
    if x is None:
        x = v.get("permissive_prediction")
    if x is None:
        return None
    try:
        x = float(x)
        return x if math.isfinite(x) else None
    except Exception:
        return None

def same_number(a, b):
    return (
        a is not None
        and b is not None
        and math.isclose(
            float(a), float(b),
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )

ORIGINAL_WRONG = {
    f["problem_id"]:
        numeric_state_from_verification(f["verification"])
    for f in FAILURES
}

assert all(v is not None for v in ORIGINAL_WRONG.values())

def transition_label(trial):
    if trial["success"]:
        return "correct"

    current = numeric_state_from_verification(
        trial["verification"]
    )
    original = ORIGINAL_WRONG[trial["problem_id"]]

    if current is None:
        return "no_numeric"

    if same_number(current, original):
        return "same_wrong"

    return "different_wrong"

for t in PILOT:
    t["_transition"] = transition_label(t)

print(
    "Pilot transitions:",
    dict(Counter(t["_transition"] for t in PILOT))
)

## 4. Action-level recurrence and repair energy

`same_wrong_given_failure` is the core recurrence quantity:

\[
P(\text{same wrong value} \mid \text{repair still failed})
\]

A high value means the intervention often falls back into the same wrong numeric state rather than merely producing some other wrong answer.

In [ ]:
ACTIONS = [
    "fresh_retry",
    "editor_rewrite",
    "tail_repair",
    "suffix_50",
    "suffix_75",
]

action_rows = []

for action in ACTIONS:
    rows = [x for x in PILOT if x["action"] == action]
    failures = [x for x in rows if not x["success"]]
    successes = [x for x in rows if x["success"]]

    p = float(np.mean([x["success"] for x in rows]))
    mean_nfe = float(np.mean([
        x["actual_forward_calls"] for x in rows
    ]))

    same_wrong_rate = float(np.mean([
        x["_transition"] == "same_wrong"
        for x in rows
    ]))

    same_wrong_given_failure = (
        float(np.mean([
            x["_transition"] == "same_wrong"
            for x in failures
        ]))
        if failures else 0.0
    )

    min_success_nfe = (
        min(x["actual_forward_calls"] for x in successes)
        if successes else math.inf
    )

    retry_energy = (
        mean_nfe / p
        if p > 0 else math.inf
    )

    action_rows.append({
        "action": action,
        "trials": len(rows),
        "success_rate": p,
        "mean_nfe": mean_nfe,
        "mean_preservation": float(np.mean([
            x["preservation"] for x in rows
        ])),
        "same_wrong_rate": same_wrong_rate,
        "same_wrong_given_failure": same_wrong_given_failure,
        "min_observed_success_nfe": min_success_nfe,
        "approx_retry_energy_nfe": retry_energy,
    })

action_energy_df = pd.DataFrame(action_rows)
display(action_energy_df)

## 5. Local-vs-global hysteresis contrast

For this analysis:

- **high-preservation local** = `tail_repair`, `suffix_75`
- **medium scope** = `suffix_50`
- **global/restart** = `fresh_retry`, `editor_rewrite`

This grouping is descriptive and specific to the AR control.

In [ ]:
LOCAL_ACTIONS = {"tail_repair", "suffix_75"}
MEDIUM_ACTIONS = {"suffix_50"}
GLOBAL_ACTIONS = {"fresh_retry", "editor_rewrite"}

def scope_of(action):
    if action in LOCAL_ACTIONS:
        return "local"
    if action in MEDIUM_ACTIONS:
        return "medium"
    if action in GLOBAL_ACTIONS:
        return "global"
    raise ValueError(action)

scope_rows = []

for scope in ["local", "medium", "global"]:
    rows = [
        x for x in PILOT
        if scope_of(x["action"]) == scope
    ]
    failed = [x for x in rows if not x["success"]]

    scope_rows.append({
        "scope": scope,
        "trials": len(rows),
        "success_rate": float(np.mean([
            x["success"] for x in rows
        ])),
        "same_wrong_rate": float(np.mean([
            x["_transition"] == "same_wrong"
            for x in rows
        ])),
        "same_wrong_given_failure": (
            float(np.mean([
                x["_transition"] == "same_wrong"
                for x in failed
            ]))
            if failed else 0.0
        ),
        "mean_nfe": float(np.mean([
            x["actual_forward_calls"] for x in rows
        ])),
        "mean_preservation": float(np.mean([
            x["preservation"] for x in rows
        ])),
    })

scope_df = pd.DataFrame(scope_rows)
display(scope_df)

local_h = float(
    scope_df.loc[
        scope_df["scope"] == "local",
        "same_wrong_given_failure"
    ].iloc[0]
)
global_h = float(
    scope_df.loc[
        scope_df["scope"] == "global",
        "same_wrong_given_failure"
    ].iloc[0]
)

print(
    "Local minus global same-wrong recurrence:",
    round(local_h - global_h, 4)
)

## 6. Per-failure repair-energy and recurrence surface

In [ ]:
surface_rows = []

for f in FAILURES:
    pid = f["problem_id"]

    for action in ACTIONS:
        rows = [
            x for x in PILOT
            if x["problem_id"] == pid
            and x["action"] == action
        ]

        assert len(rows) == 3

        failed = [x for x in rows if not x["success"]]
        p = float(np.mean([x["success"] for x in rows]))
        mean_nfe = float(np.mean([
            x["actual_forward_calls"] for x in rows
        ]))

        surface_rows.append({
            "problem_id": pid,
            "dataset": f["dataset"],
            "action": action,
            "scope": scope_of(action),
            "success_rate": p,
            "mean_nfe": mean_nfe,
            "mean_preservation": float(np.mean([
                x["preservation"] for x in rows
            ])),
            "same_wrong_rate": float(np.mean([
                x["_transition"] == "same_wrong"
                for x in rows
            ])),
            "same_wrong_given_failure": (
                float(np.mean([
                    x["_transition"] == "same_wrong"
                    for x in failed
                ]))
                if failed else 0.0
            ),
            "min_success_nfe": (
                min(
                    x["actual_forward_calls"]
                    for x in rows
                    if x["success"]
                )
                if any(x["success"] for x in rows)
                else math.inf
            ),
            "approx_retry_energy_nfe": (
                mean_nfe / p
                if p > 0 else math.inf
            ),
        })

surface_df = pd.DataFrame(surface_rows)
display(surface_df.head(15))

## 7. Exploratory pilot repair energy

For each failure, an action is considered pilot-sufficient when:
- empirical success across 3 seeds ≥ 2/3; and
- mean preservation ≥ 0.50.

The chosen action is the qualifying action with the smallest mean actual NFE.

With only 3 seeds, this is exploratory—not a certified success guarantee.

In [ ]:
PILOT_SUCCESS_THRESHOLD = 2/3
PRESERVATION_THRESHOLD = 0.50

energy_rows = []

for f in FAILURES:
    pid = f["problem_id"]
    cand = surface_df[
        (surface_df["problem_id"] == pid)
        & (
            surface_df["success_rate"]
            >= PILOT_SUCCESS_THRESHOLD
        )
        & (
            surface_df["mean_preservation"]
            >= PRESERVATION_THRESHOLD
        )
    ].copy()

    if len(cand) == 0:
        energy_rows.append({
            "problem_id": pid,
            "dataset": f["dataset"],
            "pilot_energy_found": False,
            "action": None,
            "repair_energy_nfe": math.nan,
            "success_rate": math.nan,
            "preservation": math.nan,
        })
        continue

    cand = cand.sort_values(
        ["mean_nfe", "mean_preservation"],
        ascending=[True, False],
    )

    best = cand.iloc[0]

    energy_rows.append({
        "problem_id": pid,
        "dataset": f["dataset"],
        "pilot_energy_found": True,
        "action": best["action"],
        "repair_energy_nfe": best["mean_nfe"],
        "success_rate": best["success_rate"],
        "preservation": best["mean_preservation"],
    })

energy_df = pd.DataFrame(energy_rows)

print(
    "Pilot repair-energy coverage:",
    int(energy_df["pilot_energy_found"].sum()),
    "/",
    len(energy_df),
)

display(energy_df)

## 8. Rank mechanistically informative confirmatory cases

Selection uses only the 3-seed pilot and is therefore explicitly **selection-biased**. Confirmatory estimates on these cases are mechanism probes, not unbiased population-level performance estimates.

Ranking favors:
1. high local same-wrong recurrence;
2. low local success;
3. a positive success gap for broader interventions;
4. dataset diversity.

In [ ]:
ranking_rows = []

for f in FAILURES:
    pid = f["problem_id"]

    local = surface_df[
        (surface_df["problem_id"] == pid)
        & (surface_df["scope"] == "local")
    ]
    broader = surface_df[
        (surface_df["problem_id"] == pid)
        & (surface_df["action"].isin([
            "suffix_50", "fresh_retry"
        ]))
    ]

    local_recur = float(local["same_wrong_rate"].mean())
    local_success = float(local["success_rate"].mean())
    broader_success = float(
        broader["success_rate"].max()
    )
    best_success = float(
        surface_df[
            surface_df["problem_id"] == pid
        ]["success_rate"].max()
    )

    ranking_rows.append({
        "problem_id": pid,
        "dataset": f["dataset"],
        "local_same_wrong": local_recur,
        "local_success": local_success,
        "broader_best_success": broader_success,
        "broader_minus_local":
            broader_success - local_success,
        "difficulty": 1.0 - best_success,
    })

ranking_df = pd.DataFrame(ranking_rows)

ranking_df["selection_score"] = (
    2.0 * ranking_df["local_same_wrong"]
    + 1.0 * ranking_df["broader_minus_local"]
    + 0.5 * ranking_df["difficulty"]
)

ranking_df = ranking_df.sort_values(
    [
        "selection_score",
        "local_same_wrong",
        "broader_minus_local",
    ],
    ascending=False,
)

display(ranking_df)

## 9. Select 6 cases with dataset diversity

At least one case from each available dataset is selected first, then the highest-ranked remaining cases fill the set.

In [ ]:
N_CONFIRM_FAILURES = 6

selected_ids = []

for ds in sorted(ranking_df["dataset"].unique()):
    top = ranking_df[
        ranking_df["dataset"] == ds
    ].iloc[0]["problem_id"]

    if top not in selected_ids:
        selected_ids.append(top)

for pid in ranking_df["problem_id"]:
    if pid not in selected_ids:
        selected_ids.append(pid)
    if len(selected_ids) >= N_CONFIRM_FAILURES:
        break

selected_ids = selected_ids[:N_CONFIRM_FAILURES]

CONFIRM_FAILURES = [
    FAILURE_MAP[pid]
    for pid in selected_ids
]

confirm_selection_df = ranking_df[
    ranking_df["problem_id"].isin(selected_ids)
].copy()

print("Selected confirmatory failures:", len(CONFIRM_FAILURES))
display(confirm_selection_df)

## 10. Confirmatory-run settings

Seven new seeds + the original three = **10 trials per selected case/action**.

The default adds:

6 failures × 4 actions × 7 seeds = **168 new generations**.

In [ ]:
RUN_CONFIRMATORY = True

CONFIRM_ACTIONS = [
    "tail_repair",
    "suffix_75",
    "suffix_50",
    "fresh_retry",
]

NEW_SEEDS = [
    2601, 2602, 2603, 2604, 2605, 2606, 2607
]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 96

print(
    "New confirmatory generations:",
    len(CONFIRM_FAILURES)
    * len(CONFIRM_ACTIONS)
    * len(NEW_SEEDS)
)

## 11. Save location and confirmatory run signature

Drive is used by default so a disconnect can resume the sweep.

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_CONFIRMATORY = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook06_Energy_Hysteresis"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook06_Energy_Hysteresis"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)

signature_payload = {
    "embedded_payload_sha256": EMBEDDED_PAYLOAD_SHA256,
    "selected_ids": selected_ids,
    "actions": CONFIRM_ACTIONS,
    "new_seeds": NEW_SEEDS,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_new_tokens": MAX_NEW_TOKENS,
}

CONFIRM_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

CONFIRM_PATH = OUT_DIR / "confirmatory_trials.jsonl"
MANIFEST_PATH = OUT_DIR / "run_manifest.json"

if CLEAR_OLD_CONFIRMATORY:
    for p in [CONFIRM_PATH, MANIFEST_PATH]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(encoding="utf-8")
    )
    if old["confirm_signature"] != CONFIRM_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 06 outputs use a different configuration."
        )
else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(timezone.utc).isoformat(),
                "confirm_signature": CONFIRM_SIGNATURE,
                **signature_payload,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Confirmatory signature:", CONFIRM_SIGNATURE)
print("Output:", OUT_DIR)

## 12. GPU/model preflight

This cell is skipped when `RUN_CONFIRMATORY=False`.

In [ ]:
MODEL_READY = False

if RUN_CONFIRMATORY:
    import torch
    import transformers

    os.environ[
        "PYTORCH_CUDA_ALLOC_CONF"
    ] = "expandable_segments:True"

    print("PyTorch:", torch.__version__)
    print("Transformers:", transformers.__version__)
    print("CUDA:", torch.cuda.is_available())

    assert torch.cuda.is_available(), (
        "Confirmatory extension requires a GPU runtime."
    )

    if torch.cuda.memory_allocated()/2**30 > 1.0:
        raise RuntimeError(
            "GPU is already occupied. Restart runtime before the confirmatory run."
        )

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        use_fast=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        dtype=torch.float16,
        low_cpu_mem_usage=True,
    ).to("cuda").eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    MODEL_READY = True

    print("MODEL LOAD: PASS")
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated GB:",
        round(torch.cuda.memory_allocated()/2**30, 3)
    )
else:
    print("RUN_CONFIRMATORY=False — analysis-only mode.")

## 13. Confirmatory generation helpers

In [ ]:
if RUN_CONFIRMATORY:
    class ForwardCounter:
        def __init__(self, module):
            self.module = module
            self.calls = 0
            self.handle = None

        def _hook(self, module, inputs):
            self.calls += 1

        def __enter__(self):
            self.calls = 0
            self.handle = self.module.register_forward_pre_hook(
                self._hook
            )
            return self

        def __exit__(self, exc_type, exc, tb):
            if self.handle is not None:
                self.handle.remove()
                self.handle = None

    FINAL_RE = re.compile(
        r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
        r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[eE][-+]?\d+)?)"
        r"[ \t]*$"
    )

    def verify_text(text, target):
        vals = FINAL_RE.findall(str(text))
        pred = None

        if vals:
            try:
                pred = float(vals[-1])
            except Exception:
                pred = None

        passed = (
            pred is not None
            and math.isclose(
                pred, float(target),
                abs_tol=1e-8, rel_tol=1e-8
            )
        )

        loose_vals = NUM_RE.findall(
            str(text).replace(",", "")
        )

        loose = None
        if loose_vals:
            try:
                loose = float(loose_vals[-1])
            except Exception:
                loose = None

        return {
            "passed": bool(passed),
            "strict_prediction": pred,
            "permissive_prediction": loose,
            "target": float(target),
        }

    SYSTEM = (
        "You solve quantitative reasoning problems carefully. "
        "Your final line MUST be exactly "
        "`FINAL_ANSWER: <number>` with no text after that line."
    )

    def render_problem(prompt):
        return tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM},
                {
                    "role": "user",
                    "content": (
                        prompt
                        + "\\n\\nReturn the numeric result using "
                        "the required final-answer line."
                    ),
                },
            ],
            tokenize=False,
            add_generation_prompt=True,
        )

    def preserved_ids(failed, action):
        ids = tokenizer(
            failed,
            add_special_tokens=False,
            return_tensors="pt",
        ).input_ids[0]

        if action == "suffix_50":
            return ids[:max(1, len(ids)//2)]

        if action == "suffix_75":
            return ids[:max(1, int(math.floor(
                len(ids) * 0.75
            )))]

        if action == "tail_repair":
            if "\\n" in failed:
                prefix_text = (
                    failed.rsplit("\\n", 1)[0]
                    + "\\n"
                )
                pids = tokenizer(
                    prefix_text,
                    add_special_tokens=False,
                    return_tensors="pt",
                ).input_ids[0]

                if 0 < len(pids) < len(ids):
                    return pids

            return ids[:max(
                1,
                int(math.floor(len(ids) * 0.75))
            )]

        raise ValueError(action)

    def reseed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    @torch.inference_mode()
    def generate_ids(input_ids, seed):
        reseed(seed)

        input_ids = input_ids.to(model.device)
        attention_mask = torch.ones_like(input_ids)

        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()

        with ForwardCounter(model) as fc:
            out = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        new_ids = out[0, input_ids.shape[1]:]

        return {
            "new_ids": new_ids.detach().cpu(),
            "nfe": int(fc.calls),
            "wall_seconds": float(time.time() - t0),
            "peak_vram_gb": float(
                torch.cuda.max_memory_allocated()/2**30
            ),
        }

    def run_confirm_action(failure, action, seed):
        base_text = render_problem(failure["prompt"])
        base_ids = tokenizer(
            base_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).input_ids[0]

        failed = failure["output"]

        if action == "fresh_retry":
            input_ids = base_ids.unsqueeze(0)
            generated = generate_ids(
                input_ids, seed
            )

            repaired_ids = generated["new_ids"]
            repaired = tokenizer.decode(
                repaired_ids,
                skip_special_tokens=True,
            ).strip()

            forced_fraction = 0.0

        else:
            failed_ids = tokenizer(
                failed,
                add_special_tokens=False,
                return_tensors="pt",
            ).input_ids[0]

            keep = preserved_ids(
                failed, action
            )

            input_ids = torch.cat(
                [base_ids, keep]
            ).unsqueeze(0)

            generated = generate_ids(
                input_ids, seed
            )

            repaired_ids = torch.cat([
                keep.cpu(),
                generated["new_ids"],
            ])

            repaired = tokenizer.decode(
                repaired_ids,
                skip_special_tokens=True,
            ).strip()

            forced_fraction = float(
                len(keep) / max(len(failed_ids), 1)
            )

            assert torch.equal(
                repaired_ids[:len(keep)],
                keep.cpu(),
            )

        verification = verify_text(
            repaired, failure["target"]
        )

        failed_ids_cpu = tokenizer(
            failed,
            add_special_tokens=False,
        ).input_ids

        repaired_ids_cpu = tokenizer(
            repaired,
            add_special_tokens=False,
        ).input_ids

        matcher = difflib.SequenceMatcher(
            a=failed_ids_cpu,
            b=repaired_ids_cpu,
            autojunk=False,
        )

        matched = sum(
            b.size for b in matcher.get_matching_blocks()
        )

        preservation = (
            matched / max(len(failed_ids_cpu), 1)
        )

        original_wrong = ORIGINAL_WRONG[
            failure["problem_id"]
        ]

        repaired_numeric = (
            verification["strict_prediction"]
            if verification["strict_prediction"] is not None
            else verification[
                "permissive_prediction"
            ]
        )

        if verification["passed"]:
            transition = "correct"
        elif repaired_numeric is None:
            transition = "no_numeric"
        elif same_number(
            repaired_numeric,
            original_wrong,
        ):
            transition = "same_wrong"
        else:
            transition = "different_wrong"

        return {
            "problem_id": failure["problem_id"],
            "dataset": failure["dataset"],
            "action": action,
            "seed": seed,
            "success": verification["passed"],
            "transition": transition,
            "verification": verification,
            "repaired_output": repaired,
            "preservation": float(preservation),
            "forced_prefix_fraction":
                float(forced_fraction),
            "actual_forward_calls":
                generated["nfe"],
            "wall_seconds":
                generated["wall_seconds"],
            "peak_vram_gb":
                generated["peak_vram_gb"],
            "confirm_signature":
                CONFIRM_SIGNATURE,
        }

## JSONL recovery guard

An earlier build of Notebook 06 accidentally wrote the two literal characters
`\n` between confirmatory JSON objects instead of a real newline. That can leave
a valid sequence of JSON objects in one physical line and cause
`JSONDecodeError: Extra data` on resume.

The helper below:
1. reads normal JSONL normally;
2. if necessary, incrementally decodes concatenated JSON objects;
3. preserves all already-completed trials;
4. creates a one-time `.legacy_backup`;
5. rewrites the file as proper JSONL with real newline separators.

This means already-completed GPU generations do **not** need to be discarded.

In [ ]:
def read_json_records_robust(path, repair_legacy=True):
    path = Path(path)

    if not path.exists():
        return []

    text = path.read_text(encoding="utf-8")

    if not text.strip():
        return []

    # First try ordinary JSONL.
    normal_records = []
    normal_ok = True

    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            normal_records.append(json.loads(line))
        except json.JSONDecodeError:
            normal_ok = False
            break

    if normal_ok:
        return normal_records

    # Recovery path: parse one complete JSON object at a time.
    # This is safe even when JSON string values themselves contain escaped "\\n".
    decoder = json.JSONDecoder()
    records = []
    i = 0
    n = len(text)

    while i < n:
        # Skip genuine whitespace.
        while i < n and text[i].isspace():
            i += 1

        # Skip the legacy literal backslash+n delimiter between objects.
        while text.startswith("\\n", i):
            i += 2
            while i < n and text[i].isspace():
                i += 1

        if i >= n:
            break

        try:
            obj, end = decoder.raw_decode(text, i)
        except json.JSONDecodeError as e:
            context = text[max(0, i - 80):min(n, i + 160)]
            raise RuntimeError(
                "Could not recover confirmatory JSON records. "
                f"Decoder stopped at character {i}. Context: {context!r}"
            ) from e

        records.append(obj)
        i = end

    if not records:
        raise RuntimeError("Legacy JSONL recovery found no JSON records.")

    if repair_legacy:
        backup = path.with_name(path.name + ".legacy_backup")
        if not backup.exists():
            backup.write_text(text, encoding="utf-8")

        temp = path.with_name(path.name + ".repairing")

        with open(temp, "w", encoding="utf-8") as f:
            for obj in records:
                f.write(json.dumps(obj, ensure_ascii=False))
                f.write("\n")  # REAL newline

        os.replace(temp, path)

        print(
            f"Recovered {len(records)} records and repaired {path.name}. "
            f"Backup: {backup.name}"
        )

    return records


# Repair an existing malformed confirmatory file immediately, if present.
if CONFIRM_PATH.exists():
    _existing = read_json_records_robust(
        CONFIRM_PATH,
        repair_legacy=True,
    )
    print("Existing confirmatory records readable:", len(_existing))
else:
    print("No existing confirmatory file yet; recovery guard is ready.")

## 14. Resume-safe confirmatory sweep

In [ ]:
def confirm_trial_id(pid, action, seed):
    return f"{pid}|{action}|{seed}"

def load_confirmed(path):
    done = {}

    for rec in read_json_records_robust(
        path,
        repair_legacy=True,
    ):
        if rec.get("confirm_signature") != CONFIRM_SIGNATURE:
            raise RuntimeError(
                "Existing confirmatory result uses a different configuration."
            )

        tid = rec.get("trial_id")
        if tid is None:
            raise RuntimeError(
                "A confirmatory record is missing trial_id."
            )

        done[tid] = rec

    return done

if RUN_CONFIRMATORY:
    completed = load_confirmed(
        CONFIRM_PATH
    )

    TOTAL = (
        len(CONFIRM_FAILURES)
        * len(CONFIRM_ACTIONS)
        * len(NEW_SEEDS)
    )

    print("Already completed:", len(completed), "/", TOTAL)

    with open(
        CONFIRM_PATH,
        "a",
        encoding="utf-8",
    ) as f:

        counter = 0

        for failure in CONFIRM_FAILURES:
            for action in CONFIRM_ACTIONS:
                for seed in NEW_SEEDS:
                    counter += 1

                    tid = confirm_trial_id(
                        failure["problem_id"],
                        action,
                        seed,
                    )

                    if tid in completed:
                        print(
                            f"[{counter}/{TOTAL}] SKIP {tid}"
                        )
                        continue

                    try:
                        rec = run_confirm_action(
                            failure,
                            action,
                            seed,
                        )
                        rec["trial_id"] = tid
                        rec["status"] = "ok"
                        rec["error"] = None

                    except torch.cuda.OutOfMemoryError as e:
                        torch.cuda.empty_cache()
                        gc.collect()

                        rec = {
                            "trial_id": tid,
                            "problem_id":
                                failure["problem_id"],
                            "dataset":
                                failure["dataset"],
                            "action": action,
                            "seed": seed,
                            "status": "oom",
                            "error": str(e),
                            "confirm_signature":
                                CONFIRM_SIGNATURE,
                        }

                    except Exception as e:
                        rec = {
                            "trial_id": tid,
                            "problem_id":
                                failure["problem_id"],
                            "dataset":
                                failure["dataset"],
                            "action": action,
                            "seed": seed,
                            "status": "error",
                            "error":
                                f"{type(e).__name__}: {e}",
                            "confirm_signature":
                                CONFIRM_SIGNATURE,
                        }

                    f.write(
                        json.dumps(
                            rec,
                            ensure_ascii=False,
                        )
                        + "\n"
                    )
                    f.flush()

                    if rec["status"] == "ok":
                        print(
                            f"[{counter}/{TOTAL}] "
                            f"{tid} | "
                            f"{rec['transition']} | "
                            f"NFE={rec['actual_forward_calls']}"
                        )
                    else:
                        print(
                            f"[{counter}/{TOTAL}] "
                            f"{tid} | "
                            f"{rec['status']} | "
                            f"{rec['error']}"
                        )

    print("Confirmatory sweep cell finished.")
else:
    print("Confirmatory sweep skipped.")

## 15. Combine 3 pilot + 7 new seeds for selected cases

If confirmatory generation is disabled, this section falls back to the 3-seed pilot analysis.

In [ ]:
confirm_records = []

if CONFIRM_PATH.exists():
    confirm_records = [
        x
        for x in read_json_records_robust(
            CONFIRM_PATH,
            repair_legacy=True,
        )
        if x.get("confirm_signature") == CONFIRM_SIGNATURE
    ]

if RUN_CONFIRMATORY:
    expected = (
        len(CONFIRM_FAILURES)
        * len(CONFIRM_ACTIONS)
        * len(NEW_SEEDS)
    )

    unique = {
        x["trial_id"]: x
        for x in confirm_records
    }

    print(
        "Confirmatory records:",
        len(unique),
        "/",
        expected,
    )

    bad = [
        x for x in unique.values()
        if x["status"] != "ok"
    ]

    if bad:
        raise RuntimeError(
            f"{len(bad)} confirmatory trials "
            "did not complete successfully."
        )

    if len(unique) != expected:
        raise RuntimeError(
            "Confirmatory sweep incomplete. "
            "Rerun the sweep cell."
        )

pilot_selected = [
    x for x in PILOT
    if x["problem_id"] in selected_ids
    and x["action"] in CONFIRM_ACTIONS
]

normalized_pilot = []

for x in pilot_selected:
    normalized_pilot.append({
        "problem_id": x["problem_id"],
        "dataset": x["dataset"],
        "action": x["action"],
        "seed": x["seed"],
        "success": x["success"],
        "transition": x["_transition"],
        "preservation": x["preservation"],
        "actual_forward_calls":
            x["actual_forward_calls"],
        "source": "pilot",
    })

normalized_confirm = [
    {
        "problem_id": x["problem_id"],
        "dataset": x["dataset"],
        "action": x["action"],
        "seed": x["seed"],
        "success": x["success"],
        "transition": x["transition"],
        "preservation": x["preservation"],
        "actual_forward_calls":
            x["actual_forward_calls"],
        "source": "confirmatory",
    }
    for x in confirm_records
    if x.get("status") == "ok"
]

combined_trials = (
    normalized_pilot
    + normalized_confirm
)

combined_df = pd.DataFrame(combined_trials)

print("Combined selected-case trials:", len(combined_df))

if RUN_CONFIRMATORY:
    counts = (
        combined_df.groupby(
            ["problem_id", "action"]
        ).size()
    )
    assert counts.min() == 10
    assert counts.max() == 10
    print("10-trial coverage per case/action: PASS")

## 16. Wilson intervals for 10-seed selected-case surfaces

In [ ]:
def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (math.nan, math.nan)

    p = k / n
    denom = 1 + z*z/n
    center = (
        p + z*z/(2*n)
    ) / denom

    half = (
        z
        * math.sqrt(
            p*(1-p)/n
            + z*z/(4*n*n)
        )
        / denom
    )

    return (
        max(0.0, center-half),
        min(1.0, center+half),
    )

confirm_surface_rows = []

for (pid, action), group in combined_df.groupby(
    ["problem_id", "action"]
):
    n = len(group)
    k_success = int(group["success"].sum())
    k_same = int(
        (group["transition"] == "same_wrong").sum()
    )
    failed_n = int((~group["success"]).sum())

    s_lo, s_hi = wilson(k_success, n)
    h_lo, h_hi = wilson(
        k_same,
        failed_n,
    ) if failed_n else (0.0, 0.0)

    confirm_surface_rows.append({
        "problem_id": pid,
        "dataset":
            FAILURE_MAP[pid]["dataset"],
        "action": action,
        "n": n,
        "success_rate":
            k_success / n,
        "success_ci95_low": s_lo,
        "success_ci95_high": s_hi,
        "same_wrong_given_failure":
            k_same / failed_n
            if failed_n else 0.0,
        "same_wrong_ci95_low": h_lo,
        "same_wrong_ci95_high": h_hi,
        "mean_nfe":
            float(group[
                "actual_forward_calls"
            ].mean()),
        "mean_preservation":
            float(group[
                "preservation"
            ].mean()),
    })

confirm_surface_df = pd.DataFrame(
    confirm_surface_rows
)

display(confirm_surface_df)

## 17. Confirmatory repair energy

For the selected mechanism cases, an action qualifies as a stronger pilot MSRI when:
- Wilson lower bound for success ≥ 0.50;
- mean preservation ≥ 0.50.

This is still not a universal per-instance guarantee; it is a more conservative finite-sample criterion than the 3-seed exploratory rule.

In [ ]:
confirm_energy_rows = []

for pid in selected_ids:
    cand = confirm_surface_df[
        (confirm_surface_df["problem_id"] == pid)
        & (
            confirm_surface_df[
                "success_ci95_low"
            ] >= 0.50
        )
        & (
            confirm_surface_df[
                "mean_preservation"
            ] >= 0.50
        )
    ].copy()

    if len(cand) == 0:
        confirm_energy_rows.append({
            "problem_id": pid,
            "dataset":
                FAILURE_MAP[pid]["dataset"],
            "confirmed_msri_found": False,
            "action": None,
            "repair_energy_nfe":
                math.nan,
            "success_rate": math.nan,
            "success_lcb": math.nan,
            "mean_preservation":
                math.nan,
        })
        continue

    cand = cand.sort_values(
        ["mean_nfe", "mean_preservation"],
        ascending=[True, False],
    )

    best = cand.iloc[0]

    confirm_energy_rows.append({
        "problem_id": pid,
        "dataset":
            FAILURE_MAP[pid]["dataset"],
        "confirmed_msri_found": True,
        "action": best["action"],
        "repair_energy_nfe":
            best["mean_nfe"],
        "success_rate":
            best["success_rate"],
        "success_lcb":
            best["success_ci95_low"],
        "mean_preservation":
            best["mean_preservation"],
    })

confirm_energy_df = pd.DataFrame(
    confirm_energy_rows
)

display(confirm_energy_df)

print(
    "Conservative selected-case MSRI coverage:",
    int(
        confirm_energy_df[
            "confirmed_msri_found"
        ].sum()
    ),
    "/",
    len(confirm_energy_df),
)

## 18. Selected-case hysteresis contrast

For each selected failure we compare the mean same-wrong recurrence of the two local actions (`tail_repair`, `suffix_75`) against `fresh_retry`.

In [ ]:
hysteresis_rows = []

for pid in selected_ids:
    local = confirm_surface_df[
        (confirm_surface_df["problem_id"] == pid)
        & (
            confirm_surface_df["action"].isin(
                ["tail_repair", "suffix_75"]
            )
        )
    ]

    fresh = confirm_surface_df[
        (confirm_surface_df["problem_id"] == pid)
        & (
            confirm_surface_df["action"]
            == "fresh_retry"
        )
    ].iloc[0]

    local_recur = float(
        local[
            "same_wrong_given_failure"
        ].mean()
    )

    local_success = float(
        local["success_rate"].mean()
    )

    fresh_recur = float(
        fresh["same_wrong_given_failure"]
    )

    fresh_success = float(
        fresh["success_rate"]
    )

    hysteresis_rows.append({
        "problem_id": pid,
        "dataset":
            FAILURE_MAP[pid]["dataset"],
        "local_same_wrong":
            local_recur,
        "fresh_same_wrong":
            fresh_recur,
        "recurrence_gap":
            local_recur - fresh_recur,
        "local_success":
            local_success,
        "fresh_success":
            fresh_success,
        "fresh_minus_local_success":
            fresh_success - local_success,
    })

hysteresis_df = pd.DataFrame(
    hysteresis_rows
)

display(hysteresis_df)

## 19. Hysteresis mechanism signal

This is a descriptive mechanism gate, not a hypothesis-test result.

The signal is marked present when:
- at least 2 of the 6 selected cases show a local-vs-fresh recurrence gap ≥ 0.30; and
- at least one case simultaneously shows fresh-retry success ≥ local success.

Because cases were selected from the pilot, this gate cannot be interpreted as an unbiased population prevalence estimate.

In [ ]:
strong_recurrence_cases = int(
    (
        hysteresis_df[
            "recurrence_gap"
        ] >= 0.30
    ).sum()
)

escape_cases = int(
    (
        (
            hysteresis_df[
                "recurrence_gap"
            ] >= 0.30
        )
        & (
            hysteresis_df[
                "fresh_minus_local_success"
            ] >= 0.0
        )
    ).sum()
)

HYSTERESIS_GATE = {
    "selected_cases": len(selected_ids),
    "strong_recurrence_gap_cases":
        strong_recurrence_cases,
    "recurrence_gap_threshold": 0.30,
    "cases_with_broader_escape":
        escape_cases,
    "selection_bias_warning": True,
    "HYSTERESIS_MECHANISM_SIGNAL": bool(
        strong_recurrence_cases >= 2
        and escape_cases >= 1
    ),
}

print(
    json.dumps(
        HYSTERESIS_GATE,
        indent=2,
    )
)

## 20. Save all Notebook 06 analysis artifacts

In [ ]:
OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

action_energy_path = (
    OUT_DIR / "action_repair_energy.csv"
)
scope_path = (
    OUT_DIR / "scope_hysteresis_summary.csv"
)
surface_path = (
    OUT_DIR / "pilot_energy_hysteresis_surface.csv"
)
energy_path = (
    OUT_DIR / "pilot_repair_energy.csv"
)
selection_path = (
    OUT_DIR / "confirmatory_case_selection.csv"
)
confirm_surface_path = (
    OUT_DIR / "confirmatory_surface.csv"
)
confirm_energy_path = (
    OUT_DIR / "confirmatory_repair_energy.csv"
)
hysteresis_path = (
    OUT_DIR / "confirmatory_hysteresis.csv"
)
gate_path = (
    OUT_DIR / "HYSTERESIS_GATE.json"
)

action_energy_df.to_csv(
    action_energy_path,
    index=False,
)
scope_df.to_csv(
    scope_path,
    index=False,
)
surface_df.to_csv(
    surface_path,
    index=False,
)
energy_df.to_csv(
    energy_path,
    index=False,
)
confirm_selection_df.to_csv(
    selection_path,
    index=False,
)
confirm_surface_df.to_csv(
    confirm_surface_path,
    index=False,
)
confirm_energy_df.to_csv(
    confirm_energy_path,
    index=False,
)
hysteresis_df.to_csv(
    hysteresis_path,
    index=False,
)

gate_path.write_text(
    json.dumps(
        HYSTERESIS_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved analysis outputs to:", OUT_DIR)

## 21. Create final artifact manifest

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024*1024),
            b"",
        ):
            h.update(chunk)
    return h.hexdigest()

artifact_paths = [
    MANIFEST_PATH,
    action_energy_path,
    scope_path,
    surface_path,
    energy_path,
    selection_path,
    confirm_surface_path,
    confirm_energy_path,
    hysteresis_path,
    gate_path,
]

if CONFIRM_PATH.exists():
    artifact_paths.append(
        CONFIRM_PATH
    )

artifact_manifest = {
    "created_utc":
        datetime.now(timezone.utc).isoformat(),
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "confirm_signature":
        CONFIRM_SIGNATURE,
    "files": {
        p.name: sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 22. ZIP and download all Notebook 06 results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook06_Energy_Hysteresis_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)
print(
    "Size MB:",
    round(
        Path(zip_path).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print(
            "Automatic browser download unavailable:",
            repr(e),
        )

# What to send back

Send the generated:

`RepairScopeX_Notebook06_Energy_Hysteresis_results.zip`

The most important outputs will be:

- `scope_hysteresis_summary.csv`
- `confirmatory_surface.csv`
- `confirmatory_repair_energy.csv`
- `confirmatory_hysteresis.csv`
- `HYSTERESIS_GATE.json`

If the hysteresis signal survives the extra seeds, the next notebook can move to **interventional attribution and region interaction** rather than merely repeating more AR retries.